# TFM · Notebook 04 — Interpretación y simulación de uso

**Objetivo (Fases 6 y 7 del plan):**

1. **Odds ratios con IC 95 %** de la regresión logística (statsmodels), en unidades clínicas.
2. **Importancia por permutación** del Random Forest completo.
3. **Simulación de uso**: una función que recibe un paciente ficticio y devuelve su
   probabilidad, la decisión con cada umbral y qué variables la empujan.

**Importante:** aquí no se cambia ningún modelo. Solo se interpretan los ya entrenados.

**En Colab:** `Ejecutar todas` y sube `datos_cleveland_limpios.csv` y `modelos_tfm.joblib`.

## 1. Librerías y carga

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import statsmodels.api as sm                      # regresión logística con IC y p-valores
from sklearn.inspection import permutation_importance

SEMILLA = 42
os.makedirs("figures", exist_ok=True)

try:
    from google.colab import files
    if not (glob.glob("datos_cleveland_limpios*.csv") and glob.glob("modelos_tfm*.joblib")):
        print("Sube datos_cleveland_limpios.csv y modelos_tfm.joblib:")
        files.upload()
except ImportError:
    pass

df = pd.read_csv(sorted(glob.glob("datos_cleveland_limpios*.csv"))[0])
paquete = joblib.load(sorted(glob.glob("modelos_tfm*.joblib"))[0])
import sklearn
if sklearn.__version__ != paquete["sklearn"]:
    print(f"AVISO: los modelos se guardaron con scikit-learn {paquete['sklearn']} y aquí hay "
          f"{sklearn.__version__}. Si algo falla, vuelve a ejecutar el notebook 02 en este entorno.")
modelos, umbrales = paquete["modelos"], paquete["umbrales"]

X = df.drop(columns=["num", "enfermedad", "restecg"])
y = df["enfermedad"]
X_train, y_train = X.loc[paquete["indices_train"]], y.loc[paquete["indices_train"]]
X_test, y_test = X.loc[paquete["indices_test"]], y.loc[paquete["indices_test"]]
print("Entrenamiento:", X_train.shape, "| Test:", X_test.shape)

## 2. Odds ratios con IC 95 %

**Por qué un segundo ajuste.** El modelo predictivo es la logística *penalizada* de
scikit-learn: la penalización encoge los coeficientes y no da intervalos de confianza.
Para interpretar se ajusta la misma logística **sin penalizar** con statsmodels, sobre el
mismo 80 % de entrenamiento. Así se obtienen OR con IC 95 % y p-valores, en unidades que
entiende un clínico.

Preparación (la misma lógica que el Pipeline, hecha a mano para que se lea bien):

- Perdidos de `ca` y `thal`: moda del entrenamiento.
- Continuas en unidades clínicas: OR por 10 años, por 10 mmHg, por 10 mg/dl, por 10 lpm, por 1 mm de ST y por cada vaso.
- Nominales en variables 0/1 con una **categoría de referencia**: angina típica (`cp`), pendiente ascendente (`slope`) y talio normal (`thal`).

In [ ]:
def matriz_or(datos, modas):
    d = datos.copy()
    d["ca"] = d["ca"].fillna(modas["ca"])
    d["thal"] = d["thal"].fillna(modas["thal"])
    m = pd.DataFrame(index=d.index)
    m["Edad (por 10 años)"] = d["age"] / 10
    m["Sexo masculino"] = d["sex"]
    m["Angina atípica (ref.: típica)"] = (d["cp"] == 2).astype(int)
    m["Dolor no anginoso (ref.: típica)"] = (d["cp"] == 3).astype(int)
    m["Asintomático (ref.: típica)"] = (d["cp"] == 4).astype(int)
    m["PA sistólica (por 10 mmHg)"] = d["trestbps"] / 10
    m["Colesterol (por 10 mg/dl)"] = d["chol"] / 10
    m["Glucemia > 120 mg/dl"] = d["fbs"]
    m["ECG en reposo anormal"] = d["restecg_anormal"]
    m["FC máxima (por 10 lpm)"] = d["thalach"] / 10
    m["Angina con el ejercicio"] = d["exang"]
    m["Descenso ST (por 1 mm)"] = d["oldpeak"]
    m["Pendiente ST plana (ref.: ascendente)"] = (d["slope"] == 2).astype(int)
    m["Pendiente ST descendente (ref.: ascendente)"] = (d["slope"] == 3).astype(int)
    m["Vasos coloreados (por cada vaso)"] = d["ca"]
    m["Talio: defecto fijo (ref.: normal)"] = (d["thal"] == 6).astype(int)
    m["Talio: defecto reversible (ref.: normal)"] = (d["thal"] == 7).astype(int)
    return m

modas = {"ca": X_train["ca"].mode()[0], "thal": X_train["thal"].mode()[0]}   # solo entrenamiento
M_train = matriz_or(X_train, modas)
print("Parámetros a estimar:", M_train.shape[1] + 1, "| eventos en entrenamiento:", y_train.sum())

In [ ]:
logit = sm.Logit(y_train, sm.add_constant(M_train)).fit(disp=False)
ic = np.exp(logit.conf_int())
tabla_or = pd.DataFrame({
    "OR": np.exp(logit.params),
    "IC 95 % inf": ic[0],
    "IC 95 % sup": ic[1],
    "p": logit.pvalues,
}).drop(index="const")
tabla_or["p"] = tabla_or["p"].apply(lambda p: "< 0,001" if p < 0.001 else f"{p:.3f}".replace(".", ","))
tabla_or.to_csv("figures/tabla_odds_ratios.csv")
tabla_or.round(2)

In [ ]:
# Forest plot: OR con IC 95 % en escala logarítmica
t = tabla_or.iloc[::-1]
fig, ax = plt.subplots(figsize=(7, 7))
ax.errorbar(t["OR"], range(len(t)), xerr=[t["OR"] - t["IC 95 % inf"], t["IC 95 % sup"] - t["OR"]],
            fmt="o", color="#4C72B0", ecolor="grey", capsize=3)
ax.axvline(1, color="black", lw=0.8, ls="--")
ax.set_xscale("log")
ax.set_yticks(range(len(t)), t.index, fontsize=8)
ax.set_xlabel("Odds ratio (IC 95 %, escala logarítmica)")
ax.set_title("Asociación de cada variable con enfermedad coronaria\n(logística sin penalizar, entrenamiento, n = 242)", fontsize=10)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("figures/fig_forest_odds_ratios.png", dpi=300)
plt.show()

**Cómo leerlo (y cómo no):**

- OR > 1: la variable se asocia a más probabilidad de enfermedad, **ajustando por todas las demás**.
- Si el IC cruza el 1, no hay evidencia de asociación independiente en esta muestra.
- Son asociaciones, **no causas**. Una OR "rara" (por ejemplo, colesterol sin efecto) puede deberse
  a la muestra pequeña, a la colinealidad o a que la población ya está seleccionada (remitida a angiografía).
- Con unos 111 eventos y 18 parámetros, los IC serán anchos: es honesto decirlo.

## 3. Importancia por permutación del Random Forest completo

Se desordena al azar una variable cada vez en el **test** y se mide cuánto cae el AUC.
Cuanto más cae, más depende el modelo de esa variable. Se repite 50 veces por variable.

In [ ]:
rf = modelos["Random Forest completo"]
res = permutation_importance(rf["modelo"], X_test[rf["variables"]], y_test,
                             scoring="roc_auc", n_repeats=50, random_state=SEMILLA)
NOMBRES = {"age": "Edad", "sex": "Sexo", "cp": "Tipo de dolor", "trestbps": "PA sistólica",
           "chol": "Colesterol", "fbs": "Glucemia > 120", "restecg_anormal": "ECG en reposo",
           "thalach": "FC máxima", "exang": "Angina con ejercicio", "oldpeak": "Descenso ST",
           "slope": "Pendiente ST", "ca": "Vasos coloreados", "thal": "Talio"}
imp = pd.DataFrame({"Caída media del AUC": res.importances_mean,
                    "DE": res.importances_std},
                   index=[NOMBRES[v] for v in rf["variables"]]).sort_values("Caída media del AUC")
imp.to_csv("figures/tabla_importancia_permutacion.csv")

fig, ax = plt.subplots(figsize=(6, 5))
ax.barh(imp.index, imp["Caída media del AUC"], xerr=imp["DE"], color="#55A868", ecolor="grey")
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("Caída del AUC al permutar la variable")
ax.set_title("Importancia por permutación · Random Forest completo (test)", fontsize=10)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("figures/fig_importancia_permutacion.png", dpi=300)
plt.show()

**Cuidado al interpretarla:** con 61 pacientes es ruidosa, y si dos variables aportan
información parecida (por ejemplo, angina con el ejercicio y descenso del ST) la importancia
se reparte entre ellas. Compárala con los OR: si ambos métodos señalan las mismas variables,
la conclusión es más sólida.

## 4. Simulación de uso con pacientes ficticios

Se usa el modelo final propuesto, la **logística completa** (penalizada, la del notebook 02).
La función devuelve:

- la probabilidad estimada de enfermedad coronaria;
- la decisión con el umbral 0,5 y con el umbral de *rule-out*;
- las variables que más suben o bajan la probabilidad de **ese** paciente frente al
  paciente medio del entrenamiento (contribución = coeficiente × diferencia con la media,
  en la escala del logit). Solo se muestran contribuciones apreciables (> 0,05).

In [ ]:
final = modelos["Logística completa"]
pipe = final["modelo"]
prep, lr = pipe.named_steps["prep"], pipe.named_steps["modelo"]
coef = pd.Series(lr.coef_[0], index=prep.get_feature_names_out())
# Referencia: el "paciente medio" del entrenamiento (en la escala transformada)
z_medio = pd.Series(prep.transform(X_train[final["variables"]]).mean(axis=0), index=coef.index)

ETIQUETAS = {"cp": {1.0: "angina típica", 2.0: "angina atípica", 3.0: "dolor no anginoso", 4.0: "asintomático"},
             "slope": {1.0: "pendiente ST ascendente", 2.0: "pendiente ST plana", 3.0: "pendiente ST descendente"},
             "thal": {3.0: "talio normal", 6.0: "talio: defecto fijo", 7.0: "talio: defecto reversible"}}

def nombre_legible(columna):
    grupo, var = columna.split("__", 1)
    if grupo == "nominales":
        var = var.rsplit("_", 1)[0]      # todas las columnas one-hot de una variable suman juntas
    return NOMBRES[var]

def evaluar_paciente(paciente, n_motivos=3):
    datos = pd.DataFrame([paciente])[final["variables"]]
    p = pipe.predict_proba(datos)[0, 1]
    z = pd.Series(prep.transform(datos)[0], index=coef.index)
    # Contribución frente al paciente medio: coeficiente × (valor del paciente − valor medio)
    contrib = (coef * (z - z_medio)).groupby(nombre_legible).sum().sort_values()
    # A las categóricas se les añade el valor del paciente, p. ej. "Tipo de dolor (asintomático)"
    valor = {NOMBRES[v]: f"{NOMBRES[v]} ({ETIQUETAS[v][float(paciente[v])]})" for v in ETIQUETAS}
    suben = [valor.get(n, n) for n in contrib[contrib > 0.05].index[::-1][:n_motivos]]
    bajan = [valor.get(n, n) for n in contrib[contrib < -0.05].index[:n_motivos]]
    print(f"Probabilidad estimada de enfermedad coronaria: {p:.0%}")
    print(f"  Umbral 0,5:                 {'POSITIVO' if p >= 0.5 else 'negativo'}")
    print(f"  Umbral rule-out ({umbrales['Logística completa']:.2f}):     "
          f"{'positivo: no descartar' if p >= umbrales['Logística completa'] else 'NEGATIVO: probabilidad baja'}")
    print("  Frente al paciente medio, SUBEN la probabilidad:", ", ".join(suben) or "ninguna destacable")
    print("  Frente al paciente medio, la BAJAN:             ", ", ".join(bajan) or "ninguna destacable")
    print("  ⚠️ Herramienta de apoyo. No sustituye el juicio clínico ni las pruebas indicadas.")
    return p

In [ ]:
pacientes = {
    "A · Mujer de 45 años, dolor no anginoso, pruebas normales": dict(
        age=45, sex=0, cp=3, trestbps=125, chol=210, fbs=0, restecg_anormal=0,
        thalach=170, exang=0, oldpeak=0.0, slope=1, ca=0, thal=3),
    "B · Varón de 58 años, angina atípica, pruebas dudosas": dict(
        age=58, sex=1, cp=2, trestbps=140, chol=250, fbs=0, restecg_anormal=1,
        thalach=145, exang=0, oldpeak=1.0, slope=2, ca=1, thal=3),
    "C · Varón de 62 años, asintomático, ergometría y talio positivos": dict(
        age=62, sex=1, cp=4, trestbps=150, chol=280, fbs=1, restecg_anormal=1,
        thalach=120, exang=1, oldpeak=2.5, slope=2, ca=2, thal=7),
}
for titulo, datos_paciente in pacientes.items():
    print("\n" + titulo)
    evaluar_paciente(datos_paciente)

**Para la memoria (capítulo 5):** muestra estos tres casos y discute sus límites.
1. El modelo se entrenó con pacientes **remitidos a angiografía** en los años 80: no sirve para cribado ni para urgencias.
2. Usa variables que se obtienen tras ergometría, talio y fluoroscopia: su lugar es al final del proceso diagnóstico, no en la primera consulta.
3. Como sistema de apoyo al diagnóstico en la UE sería un **producto sanitario** y un **sistema de IA de alto riesgo**: supervisión humana, transparencia, monitorización del rendimiento (*data drift*) y validación externa antes de cualquier uso real. Cita el texto oficial de los reglamentos.

## 5. Versiones del software (para el apartado 2.11 de la memoria)

In [ ]:
import sys, scipy, matplotlib, statsmodels
print("Python", sys.version.split()[0])
for lib in (pd, np, scipy, sklearn, statsmodels, matplotlib, joblib):
    print(f"{lib.__name__:<12} {lib.__version__}")

## 6. Guardar resultados

In [ ]:
try:
    from google.colab import files
    import shutil
    shutil.make_archive("resultados_interpretacion", "zip", "figures")
    files.download("resultados_interpretacion.zip")
except ImportError:
    print("Tablas y figuras guardadas en figures/")